In [ ]:
import mlflow
import pandas as pd
import matplotlib.pyplot as plt
import re
import numpy as np

# Set the MLflow tracking URI to your database
mlflow.set_tracking_uri('sqlite:///sslbirds.db')

# Get the experiment by name
experiment_name = 'eat_hsn'
experiment = mlflow.get_experiment_by_name(experiment_name)

In [ ]:
import mlflow

# Set the MLflow tracking URI to your database (go up one directory)
mlflow.set_tracking_uri('sqlite:///../sslbirds.db')

# Get all experiments
experiments = mlflow.search_experiments()

print("Experiment details:")
print("-" * 60)
for exp in experiments:
    # Count runs in this experiment
    runs = mlflow.search_runs(experiment_ids=[exp.experiment_id])
    run_count = len(runs)
    
    print(f"Experiment: {exp.name}")
    print(f"  ID: {exp.experiment_id}")
    print(f"  Runs: {run_count}")
    print(f"  Status: {exp.lifecycle_stage}")
    print()

In [ ]:
import mlflow
import pandas as pd
import matplotlib.pyplot as plt
import re
import numpy as np

# Set the MLflow tracking URI to your database
mlflow.set_tracking_uri('sqlite:///../sslbirds.db')

# Get the experiment by name
experiment_name = 'eat_hsn'
experiment = mlflow.get_experiment_by_name(experiment_name)

if experiment is None:
    print(f"Experiment '{experiment_name}' not found!")
    # List all available experiments
    print("Available experiments:")
    for exp in mlflow.search_experiments():
        print(f"  - {exp.name} (ID: {exp.experiment_id})")
else:
    print(f"Found experiment: {experiment.name} (ID: {experiment.experiment_id})")
    
    # Search for runs in the experiment that match the pattern (only 2025-08-03)
    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string="tags.mlflow.runName LIKE 'eat_hsn_HSN_2025-08-03_%'"
    )
    
    print(f"Found {len(runs)} runs")
    
    if len(runs) > 0:
        # Process the data
        results = []
        
        for _, run in runs.iterrows():
            run_name = run['tags.mlflow.runName']
            
            # Extract epoch from pretrained_weights_path parameter
            pretrained_path = run.get('params.module/network/pretrained_weights_path', '')
            
            if pretrained_path:
                # Extract epoch number from path like "...epoch=04.ckpt" or "...epoch=34.ckpt"
                epoch_match = re.search(r'epoch=(\d+)\.ckpt', pretrained_path)
                if epoch_match:
                    epoch = int(epoch_match.group(1))
                    
                    # Get metrics - try different possible metric names
                    test_map = (run.get('metrics.test_MultilabelAveragePrecision') or 
                               run.get('metrics.test_map') or 
                               run.get('metrics.test_MAP'))
                    
                    test_auroc = (run.get('metrics.test_MultilabelAUROC') or 
                                 run.get('metrics.test_auroc') or 
                                 run.get('metrics.test_AUROC'))
                    
                    test_top1 = (run.get('metrics.test_TopKAccuracy') or 
                                run.get('metrics.test_top1_acc') or 
                                run.get('metrics.test_top1'))
                    
                    results.append({
                        'run_name': run_name,
                        'epoch': epoch,
                        'test_map': test_map,
                        'test_auroc': test_auroc, 
                        'test_top1_acc': test_top1
                    })
        
        # Convert to DataFrame and sort by epoch
        results_df = pd.DataFrame(results)
        
        if len(results_df) > 0:
            results_df = results_df.sort_values('epoch')
            
            print(f"\nProcessed {len(results_df)} experiments with valid data")
            print("\nResults:")
            print(results_df[['epoch', 'test_map', 'test_auroc', 'test_top1_acc']].round(4))
            
            # Create plots
            fig, axes = plt.subplots(1, 3, figsize=(18, 5))
            
            # Plot 1: MAP
            valid_map = results_df['test_map'].notna()
            if valid_map.any():
                axes[0].plot(results_df[valid_map]['epoch'], results_df[valid_map]['test_map'], 
                           'o-', linewidth=2, markersize=8, color='blue')
            axes[0].set_xlabel('Pretraining Epochs', fontsize=12)
            axes[0].set_ylabel('Test MAP', fontsize=12)
            axes[0].set_title('Mean Average Precision vs Pretraining Epochs', fontsize=14)
            axes[0].grid(True, alpha=0.3)
            if valid_map.any():
                axes[0].set_xticks(results_df[valid_map]['epoch'])
            
            # Plot 2: AUROC
            valid_auroc = results_df['test_auroc'].notna()
            if valid_auroc.any():
                axes[1].plot(results_df[valid_auroc]['epoch'], results_df[valid_auroc]['test_auroc'], 
                           'o-', color='orange', linewidth=2, markersize=8)
            axes[1].set_xlabel('Pretraining Epochs', fontsize=12)
            axes[1].set_ylabel('Test AUROC', fontsize=12)
            axes[1].set_title('AUROC vs Pretraining Epochs', fontsize=14)
            axes[1].grid(True, alpha=0.3)
            if valid_auroc.any():
                axes[1].set_xticks(results_df[valid_auroc]['epoch'])
            
            # Plot 3: Top-1 Accuracy  
            valid_top1 = results_df['test_top1_acc'].notna()
            if valid_top1.any():
                axes[2].plot(results_df[valid_top1]['epoch'], results_df[valid_top1]['test_top1_acc'], 
                           'o-', color='green', linewidth=2, markersize=8)
            axes[2].set_xlabel('Pretraining Epochs', fontsize=12)
            axes[2].set_ylabel('Test Top-1 Accuracy', fontsize=12)
            axes[2].set_title('Top-1 Accuracy vs Pretraining Epochs', fontsize=14)
            axes[2].grid(True, alpha=0.3)
            if valid_top1.any():
                axes[2].set_xticks(results_df[valid_top1]['epoch'])
            
            plt.tight_layout()
            plt.show()
            
            # Show best results
            print("\n" + "="*50)
            print("BEST RESULTS:")
            print("="*50)
            
            if results_df['test_map'].notna().any():
                best_map_idx = results_df['test_map'].idxmax()
                print(f"Best MAP: {results_df.loc[best_map_idx, 'test_map']:.4f} at epoch {results_df.loc[best_map_idx, 'epoch']}")
            
            if results_df['test_auroc'].notna().any():
                best_auroc_idx = results_df['test_auroc'].idxmax()
                print(f"Best AUROC: {results_df.loc[best_auroc_idx, 'test_auroc']:.4f} at epoch {results_df.loc[best_auroc_idx, 'epoch']}")
            
            if results_df['test_top1_acc'].notna().any():
                best_top1_idx = results_df['test_top1_acc'].idxmax()
                print(f"Best Top-1: {results_df.loc[best_top1_idx, 'test_top1_acc']:.4f} at epoch {results_df.loc[best_top1_idx, 'epoch']}")
                
        else:
            print("No valid results found after processing.")
            print("Available columns in runs:")
            print(runs.columns.tolist())
    else:
        print("No runs found matching the pattern 'eat_hsn_HSN_%'")
        
        # Debug: show all runs in the experiment
        all_runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
        print(f"Total runs in experiment: {len(all_runs)}")
        if len(all_runs) > 0:
            print("Available run names:")
            for name in all_runs['tags.mlflow.runName']:
                print(f"  - {name}")